In [1]:
import torch
from torch import nn, optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.metrics import davies_bouldin_score
from sklearn.decomposition import PCA
from scipy.stats import chi2
from sklearn.mixture import GaussianMixture
from collections import deque, defaultdict

# CAE

In [2]:
train = pd.read_csv('../../../train_CADE_cod.csv')
val = pd.read_csv('../../../val_CADE_cod.csv')
test = pd.read_csv('../../../test_CADE_cod.csv')

In [3]:
classes_out = ['Infilteration','DoS attacks-SlowHTTPTest']
train = train.drop(index=train[train['Label'].isin(classes_out)].index)
val = val.drop(index=val[val['Label'].isin(classes_out)].index)
test = test.drop(index=test[test['Label'].isin(classes_out)].index)

# CAE CADE (margin 1 e lambda 1.0)

In [4]:
def pick_pair(embeddings, labels, pick_embeddings, pick_labels):
    """
    Gera todos os pares entre embeddings e pick_embeddings,
    com labels_pair indicando se os pares são positivos (0) ou negativos (1).
    """
    N, D = embeddings.shape
    M = pick_embeddings.shape[0]

    # Expande para todas as combinações possíveis
    A = embeddings.unsqueeze(0).expand(M, N, D).reshape(M * N, D)
    B = pick_embeddings.unsqueeze(1).expand(M, N, D).reshape(M * N, D)

    labels_exp = labels.unsqueeze(0).expand(M, N)
    pick_labels_exp = pick_labels.unsqueeze(1).expand(M, N)
    labels_pair = (labels_exp != pick_labels_exp).reshape(-1).float()

    return A, B, labels_pair

def ContrastiveLoss(input1, input2, y, margin=1.0):
    # y = 0 quando input1 e input2 são da mesma classe e y = 1 caso contrário
    diff = input1 - input2
    dist_sq = torch.sum(torch.pow(diff,2),1)
    dist = torch.sqrt(dist_sq + 1e-10) # Soma 1e-10 pois se o sqrt der 0 o gradiente da NaN
    mdist = margin - dist
    mdist = F.relu(mdist)
    loss = ((1-y) * dist * dist) + (y * mdist * mdist)
    return torch.mean(loss)

def AEContrastive(input, output, embeddings, labels, pick_encoded, pick_labels, margin=1.0, lambda1=1.0):
    mseloss = nn.MSELoss()
    ae_loss = mseloss(input,output)

    A, B, labels_pair = pick_pair(embeddings, labels, pick_encoded, pick_labels)
    contrastive_loss = ContrastiveLoss(A, B, labels_pair, margin=margin)
    return ae_loss + contrastive_loss * lambda1, ae_loss, contrastive_loss

In [5]:
class contrastive_ae(nn.Module):
    def __init__(self, input_neurons, neurons):
        super().__init__()
        self.encoder0 = nn.Linear(input_neurons,64)
        self.encoder1 = nn.Linear(64,32)
        self.encoder2 = nn.Linear(32,16)
        self.encoder3 = nn.Linear(16,neurons)

        self.decoder0 = nn.Linear(neurons,16)
        self.decoder1 = nn.Linear(16,32)
        self.decoder2 = nn.Linear(32,64)
        self.decoder3 = nn.Linear(64,input_neurons)

        self.activation0 = nn.ReLU()
    
    def forward(self, X):
        X = self.activation0(self.encoder0(X))
        X = self.activation0(self.encoder1(X))
        X = self.activation0(self.encoder2(X))
        X = self.encoder3(X)
        encoded = X
        X = self.activation0(self.decoder0(X))
        X = self.activation0(self.decoder1(X))
        X = self.activation0(self.decoder2(X))
        X = self.decoder3(X)


        return X, encoded

In [6]:
class encoder(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.encoder0 = list(model.children())[0]
        self.encoder1 = list(model.children())[1]
        self.encoder2 = list(model.children())[2]
        self.encoder3 = list(model.children())[3]
        self.activation0 = list(model.children())[8]
    
    def forward(self, X):
        X = self.activation0(self.encoder0(X))
        X = self.activation0(self.encoder1(X))
        X = self.activation0(self.encoder2(X))
        X = self.encoder3(X)
        return X

In [7]:
from torch.utils.data import Dataset, DataLoader,TensorDataset
import random
def build_class_reference_batches(X_train, y_train, batch_size=512):
    classes = torch.unique(y_train).tolist()
    n_classes = len(classes)
    total_samples = len(X_train)
    dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    

    print(classes)

    # Mapeia cada classe para seus índices
    class_to_indices = {
        int(cls.item()): (y_train == cls).nonzero(as_tuple=True)[0].tolist()
        for cls in torch.unique(y_train)
    }

    reference_list = []
    total_batches = len(train_loader)

    for batch_idx in range(total_batches):
        start = batch_idx * batch_size
        end = min(start + batch_size, total_samples)
        batch_indices = set(range(start, end))

        class_samples = []
        for cls in classes:
            candidates = [i for i in class_to_indices[cls] if i not in batch_indices]
            if not candidates:
                raise ValueError(f"Classe {cls} não possui amostras fora do batch {batch_idx}")
            chosen = candidates[torch.randint(len(candidates), (1,)).item()]
            class_samples.append(chosen)

        reference_list.append(class_samples)


    return train_loader, reference_list

In [8]:
def normalize(t):
    t_norm = (t - t.mean()) / t.std()
    return t_norm


In [9]:
train['Label'] = train['Label'].replace('FTP-BruteForce','BruteForce')
train['Label'] = train['Label'].replace('SSH-Bruteforce','BruteForce')
train['Label'] = train['Label'].replace('DoS attacks-GoldenEye','DoS')
train['Label'] = train['Label'].replace('DoS attacks-Slowloris','DoS')
## train['Label'] = train['Label'].replace('DoS attacks-SlowHTTPTest','DoS')
train['Label'] = train['Label'].replace('DoS attacks-Hulk','DoS')
# train['Label'] = train['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
train['Label'] = train['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
train['Label'] = train['Label'].replace('DDOS attack-HOIC','DDoS')
train['Label'] = train['Label'].replace('Brute Force -Web','Web')
train['Label'] = train['Label'].replace('Brute Force -XSS','Web')
train['Label'] = train['Label'].replace('SQL Injection','Web')


val['Label'] = val['Label'].replace('FTP-BruteForce','BruteForce')
val['Label'] = val['Label'].replace('SSH-Bruteforce','BruteForce')
val['Label'] = val['Label'].replace('DoS attacks-GoldenEye','DoS')
val['Label'] = val['Label'].replace('DoS attacks-Slowloris','DoS')
## val['Label'] = val['Label'].replace('DoS attacks-SlowHTTPTest','DoS')
val['Label'] = val['Label'].replace('DoS attacks-Hulk','DoS')
# val['Label'] = val['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
val['Label'] = val['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
val['Label'] = val['Label'].replace('DDOS attack-HOIC','DDoS')
val['Label'] = val['Label'].replace('Brute Force -Web','Web')
val['Label'] = val['Label'].replace('Brute Force -XSS','Web')
val['Label'] = val['Label'].replace('SQL Injection','Web')


test['Label'] = test['Label'].replace('FTP-BruteForce','BruteForce')
test['Label'] = test['Label'].replace('SSH-Bruteforce','BruteForce')
test['Label'] = test['Label'].replace('DoS attacks-GoldenEye','DoS')
test['Label'] = test['Label'].replace('DoS attacks-Slowloris','DoS')
## test['Label'] = test['Label'].replace('DoS attacks-SlowHTTPTest','DoS')
test['Label'] = test['Label'].replace('DoS attacks-Hulk','DoS')
# test['Label'] = test['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
test['Label'] = test['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
test['Label'] = test['Label'].replace('DDOS attack-HOIC','DDoS')
test['Label'] = test['Label'].replace('Brute Force -Web','Web')
test['Label'] = test['Label'].replace('Brute Force -XSS','Web')
test['Label'] = test['Label'].replace('SQL Injection','Web')

In [10]:
labels_str = [i for i in train['Label'].value_counts().index]
print(labels_str)

for i, l in enumerate(labels_str):
    train['Label'] = train['Label'].replace(l, i)
    val['Label'] = val['Label'].replace(l, i)
    test['Label'] = test['Label'].replace(l, i)

['Benign', 'DDoS', 'DDoS attacks-LOIC-HTTP', 'DoS', 'BruteForce', 'Bot', 'Web']


/tmp/ipykernel_776206/2307520911.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train['Label'] = train['Label'].replace(l, i)
/tmp/ipykernel_776206/2307520911.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  val['Label'] = val['Label'].replace(l, i)
/tmp/ipykernel_776206/2307520911.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('f

In [11]:
array_hidden_classes = [[2]]
filenames = ['2']
# array_hidden_classes = [[0]]
# filenames = ['0']
total_classes = len(labels_str)
for i in range(len(array_hidden_classes)):
    filename = f'{filenames[i]}_hidden'
    print(filename)
    hidden_classes = array_hidden_classes[i] #Classes ocultas do treinamento

    hidden_classes_index = list(train[train['Label'].isin(hidden_classes)].index)

    train_hidden = train.loc[hidden_classes_index]

    train_not_hidden = train.drop(hidden_classes_index)

    X_train = train_not_hidden.drop(columns=['Label'])
    y_train = train_not_hidden['Label']
    X_val = val.drop(columns=['Label'])
    y_val = val['Label'].values
    X_test = test.drop(columns=['Label'])
    y_test = test['Label'].values

    torch.manual_seed(123)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(device)
    X_train = torch.tensor(np.array(X_train), dtype=torch.float)
    y_train = torch.tensor(np.array(y_train), dtype=torch.float)

    train_loader, pick_list = build_class_reference_batches(X_train,y_train, batch_size=512)
    pick_samples = []
    pick_labels = []
    for l in pick_list:
        pick_samples.append(X_train[l])
        pick_labels.append(y_train[l])

    pick_samples = torch.stack(pick_samples)    
    pick_labels = torch.stack(pick_labels)

    print('pick completo')

    neurons = total_classes - len(hidden_classes)
    print(f'{X_train.shape[1]} {neurons}')
    model = contrastive_ae(input_neurons=X_train.shape[1],neurons=neurons)
    model.to(device)

    learning_rate = 0.0001
    opt = optim.Adam(model.parameters(),lr=learning_rate)
    margin = 10 # Parametro de afastamento das classes
    cae_lambda_1 = 0.1 # Parametro de influencia na distancia das classes
    
    
    for epoch in range(250):
        running_loss = 0
        running_ael = 0
        running_cl = 0
        contador = 0
        for data in train_loader:
            inputs, labels = data
            inputs = inputs.to(device)
            labels = labels.to(device)
            ps = pick_samples[contador]
            pl = pick_labels[contador]
            ps = ps.to(device)
            pl = pl.to(device)
            contador += 1
            #print(f'{contador}/{len(train_loader)}')
            opt.zero_grad()
            outputs, encoded = model(inputs)
            _, pick_encoded = model(ps)
            tam_pe = pick_encoded.shape[0]
            enc_total = torch.cat((encoded,pick_encoded), dim=0)
            enc_total_norm = normalize(enc_total)
            encoded_norm = enc_total_norm[:-tam_pe]
            pick_encoded_norm = enc_total_norm[-tam_pe:]
            loss, ael, cl = AEContrastive(input=inputs, output=outputs, embeddings = encoded_norm, labels=labels, pick_encoded = pick_encoded_norm, pick_labels = pl, margin=margin, lambda1=cae_lambda_1)
            #loss = dummyMSELoss(inputs,outputs)
            loss.backward()
            opt.step()
            running_loss += loss.item()
            running_ael += ael.item()
            running_cl += cl.item()
            #print(f'loss = {loss.item()} batch_size = {size}')
        print(f'filename: {filename} Epoch {epoch+1} loss:{running_loss/len(train_loader)} ael:{running_ael/len(train_loader)} cl:{running_cl/len(train_loader)}')

    model_e = encoder(model=model)
    model_e.to(device)

    model_e.eval()

    X_train = torch.tensor(np.array(X_train), dtype=torch.float)
    y_train = torch.tensor(np.array(y_train), dtype=torch.float)

    train_encoded = model_e(X_train.to(device))

    train_encoded = train_encoded.detach().cpu().numpy()

    train_encoded_df = pd.DataFrame(train_encoded)
    train_encoded_df['Label'] = y_train.detach().cpu().numpy().astype(int)
    display(train_encoded_df)

    score = davies_bouldin_score(train_encoded_df.drop(columns=['Label']), train_encoded_df['Label'])
    print("Davies-Bouldin Score:", score)

    X_val = torch.tensor(np.array(X_val), dtype=torch.float)

    val_encoded = model_e(X_val.to(device))

    val_encoded = val_encoded.detach().cpu().numpy()

    val_encoded_df = pd.DataFrame(val_encoded)

    val_encoded_df['Label'] = y_val

    display(val_encoded_df)

    X_test = torch.tensor(np.array(X_test), dtype=torch.float)

    test_encoded = model_e(X_test.to(device))

    test_encoded = test_encoded.detach().cpu().numpy()

    test_encoded_df = pd.DataFrame(test_encoded)

    test_encoded_df['Label'] = y_test

    display(test_encoded_df)

    train_encoded_df.to_csv(f'train_encoded_CADE_cod_pick_class_squared_normalized_{filename}.csv',index=False)
    val_encoded_df.to_csv(f'val_encoded_CADE_cod_pick_class_squared_normalized_{filename}.csv',index=False)
    test_encoded_df.to_csv(f'test_encoded_CADE_cod_pick_class_squared_normalized_{filename}.csv',index=False)

2_hidden


cuda


[0.0, 1.0, 3.0, 4.0, 5.0, 6.0]


pick completo
82 6


filename: 2_hidden Epoch 1 loss:3.3092190643615313 ael:0.02366287424020081 cl:32.85556140764968


...

/tmp/ipykernel_776206/775940180.py:91: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  X_train = torch.tensor(np.array(X_train), dtype=torch.float)
/tmp/ipykernel_776206/775940180.py:92: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  y_train = torch.tensor(np.array(y_train), dtype=torch.float)


,0,1,2,3,4,5,Label
0,-0.089919,-0.096427,-0.090080,-0.072012,-0.072005,-0.073633,0
1,-0.091263,-0.069573,-0.075672,-0.081091,-0.104401,-0.079564,1
2,-0.053573,-0.068667,-0.080397,-0.082585,-0.075255,-0.058829,3
3,-0.087867,-0.095344,-0.088023,-0.068918,-0.074882,-0.071806,0
4,-0.090250,-0.076230,-0.083614,-0.122893,-0.057831,-0.100551,5
...,...,...,...,...,...,...,...
1711088,-0.091318,-0.069574,-0.075687,-0.081129,-0.104440,-0.079540,1
1711089,-0.053577,-0.068670,-0.080398,-0.082580,-0.075260,-0.058825,3
1711090,-0.053580,-0.068673,-0.080399,-0.082575,-0.075265,-0.058822,3
1711091,-0.091233,-0.069572,-0.075664,-0.081070,-0.104380,-0.079577,1


Davies-Bouldin Score: 0.723211801292227


,0,1,2,3,4,5,Label
0,-0.095232,-0.116997,-0.092246,-0.035714,-0.081262,-0.062000,2
1,-0.053561,-0.068658,-0.080394,-0.082605,-0.075235,-0.058841,3
2,-0.091553,-0.099775,-0.089056,-0.068791,-0.073355,-0.072392,0
3,-0.066241,-0.076682,-0.062796,-0.058205,-0.072155,-0.111649,4
4,-0.090403,-0.099144,-0.089511,-0.069095,-0.073257,-0.072288,0
...,...,...,...,...,...,...,...
519951,-0.089577,-0.097037,-0.088007,-0.071075,-0.073065,-0.073324,0
519952,-0.090053,-0.077536,-0.082630,-0.121387,-0.058328,-0.101108,5
519953,-0.091304,-0.069574,-0.075683,-0.081120,-0.104430,-0.079546,1
519954,-0.089782,-0.096598,-0.089365,-0.071380,-0.071588,-0.072641,0


,0,1,2,3,4,5,Label
0,-0.091269,-0.069573,-0.075673,-0.081095,-0.104405,-0.079561,1
1,-0.095450,-0.117289,-0.092246,-0.035487,-0.081220,-0.062447,2
2,-0.089952,-0.099437,-0.089079,-0.067146,-0.072995,-0.069843,0
3,-0.089982,-0.098973,-0.088248,-0.067915,-0.072353,-0.072656,0
4,-0.090903,-0.069362,-0.074746,-0.081843,-0.105543,-0.079578,1
...,...,...,...,...,...,...,...
649942,-0.090272,-0.098792,-0.089353,-0.069344,-0.073223,-0.072355,0
649943,-0.090843,-0.069088,-0.074701,-0.082195,-0.105493,-0.079393,1
649944,-0.090888,-0.069198,-0.074736,-0.082076,-0.105544,-0.079398,1
649945,-0.067365,-0.078022,-0.063194,-0.057313,-0.072076,-0.111074,4
